In [1]:
import pandas as pd
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

In [2]:
df = pd.read_csv("../data/creditcard.csv")

df['Hour'] = (df['Time'] // 3600) % 24

X = df.drop(columns=['Time', 'Class'])
y = df['Class']

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size = 0.8, random_state = 42, stratify = y)

In [4]:
robust_col = ['Amount']
std_col = ['Hour']

preprocessor = ColumnTransformer(transformers=[('amount', RobustScaler(), robust_col),
                                             ('hour', StandardScaler(), std_col)],
                               remainder='passthrough')

In [5]:
#LOGISTIC REGRESSION
pipeline_lr = ImbPipeline(steps=[('preprocessor', preprocessor),
                           ('smote', SMOTE(random_state=42)),
                           ('model', LogisticRegression(C=0.1, max_iter=1000))])
pipeline_lr.fit(X_train, y_train)
y_pred = pipeline_lr.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.97      0.99     56864
           1       0.06      0.92      0.11        98

    accuracy                           0.97     56962
   macro avg       0.53      0.95      0.55     56962
weighted avg       1.00      0.97      0.98     56962



In [6]:
#RANDOM FOREST CLASSIFIER
from sklearn.ensemble import RandomForestClassifier
pipeline_rf = ImbPipeline(steps=[('preprocessor', preprocessor),
                           ('model', RandomForestClassifier(random_state=42, n_jobs=-1, max_depth=10))])
pipeline_rf.fit(X_train, y_train)
y_pred = pipeline_rf.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.94      0.82      0.87        98

    accuracy                           1.00     56962
   macro avg       0.97      0.91      0.94     56962
weighted avg       1.00      1.00      1.00     56962



In [7]:
#SAVE MODEL
import joblib

model_path = '../models/lr_model.pkl'

joblib.dump(pipeline_lr, model_path)

['../models/lr_model.pkl']

In [8]:
#EXPLAINER
import shap

lr_model = pipeline_lr.named_steps['model']
lr_preprocessor = pipeline_lr.named_steps['preprocessor']

X_train_rescaled  = lr_preprocessor.transform(X_train)

explainer = shap.LinearExplainer(lr_model, X_train_rescaled)

explainer_path = "../models/lr_explainer.pkl"
joblib.dump(explainer, explainer_path)


['../models/lr_explainer.pkl']